In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('../../data/raw/housing-infomation.csv')
df

,ข้อมูลที่อยู่อาศัยสร้างเสร็จจดทะเบียน เดือน พฤษภาคม 2569,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,การแสดงผลจังหวัด แยกตามอำเภอ (กรุงเทพมหานคร),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ลำดับที่,จังหวัด/อำเภอ,บ้าน (1),โรงแรม (7),โรงงาน (13),หอพัก (14),ตึกแถว (16),ห้องแถว (17),ตึก (18),ร้านค้า (19),...,ห้างหุ้นส่วนจำกัด (33),ห้างหุ้นส่วนนิติบุคคล (34),โกดัง (37),บริษัท (38),ทาวน์เฮ้าส์ (39),บ้านแฝด (45),บ้านแถว (49),ทาวน์โฮม (51),อื่น ๆ,รวม
4,NaN,รวม,"1,969,244","1,778",656,"2,193","52,671","1,057","106,609","12,488",...,23,5,792,558,"101,000","22,662","38,586",765,"107,746","3,373,689"
5,NaN,กรุงเทพมหานคร,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1,ท้องถิ่นเขตพระนคร,"16,602",116,0,2,11,0,16,775,...,0,0,0,1,0,0,0,0,697,"18,771"
7,2,ท้องถิ่นเขตดุสิต,"19,570",4,9,6,110,21,431,4,...,0,0,0,1,23,0,0,0,"9,175","34,408"
8,3,ท้องถิ่นเขตหนองจอก,"53,799",1,23,12,774,0,"12,090",250,...,0,0,52,8,"3,154","2,079",506,0,"1,993","76,434"
9,4,ท้องถิ่นเขตบางรัก,"20,250",112,0,4,369,17,"5,236",48,...,0,0,0,0,0,0,0,0,361,"36,130"


In [ ]:
import pandas as pd
import json
import os
import gc

# 1. กำหนดเส้นทางไฟล์
raw_file_path = "../../data/raw/housing-infomation.csv"
output_file_path = "../../data/interim/bangkok_housing.json"

print("1. ⚙️ กำลังโหลดข้อมูลสิ่งปลูกสร้างจดทะเบียน (ปรับ skiprows=4 เพื่อแก้บั๊ก KeyError)...")
# แก้ไขจาก skiprows=5 เป็น skiprows=4 เพื่อชี้ตำแหน่งไปที่แถว Header หลักของจริงที่มีคำว่า 'ลำดับที่' พอดี
df_raw = pd.read_csv(raw_file_path, skiprows=4)

print("2. 🧹 กำลังทำความสะอาดล้างแถวสรุปยอดและล้างชื่อเขตให้เป็นมาตรฐาน...")
# ลบแถวที่เป็นค่ายอดรวมทั้งหมด (ดักจับจากแถวที่ 'ลำดับที่' เป็นค่าว่าง)
df_filtered = df_raw.dropna(subset=['ลำดับที่']).copy()

# ล้างคำว่า "ท้องถิ่นเขต" ออกให้เหลือแค่คำว่า "เขต..." เพื่อให้ง่ายต่อการนำไป Join เชิงพื้นที่กับ JSON อื่น
df_filtered['จังหวัด/อำเภอ'] = df_filtered['จังหวัด/อำเภอ'].astype(str).str.replace('ท้องถิ่นเขต', 'เขต').str.strip()

print("3. 🔢 แปลงฟีเจอร์ตัวเลข String (ที่มีคอมมา) ให้กลายเป็นประเภท Integer...")
# เลือกสกัดเฉพาะฟีเจอร์สำคัญเชิงธุรกิจทำเลร้านอาหาร/คาเฟ่
target_features = {
    'จังหวัด/อำเภอ': 'district',
    'บ้าน (1)': 'house',
    'ทาวน์เฮ้าส์ (39)': 'townhouse',
    'บ้านแฝด (45)': 'semi_detached_house',
    'อาคารชุด (22)': 'condo',
    'แฟลต (21)': 'flat',
    'หอพัก (14)': 'dormitory',
    'สำนักงาน (20)': 'office',
    'ร้านค้า (19)': 'shop',
    'ตึกแถว (16)': 'shophouse',
    'รวม': 'total_buildings'
}

df_final = pd.DataFrame()

# วนลูปคลีนเครื่องหมายคอมมา "," และแปลงเดต้าเป็นตัวเลขถ้วนๆ ป้องกันปัญหาคณิตศาสตร์พัง
for raw_col, new_col in target_features.items():
    if raw_col in df_filtered.columns:
        if raw_col == 'จังหวัด/อำเภอ':
            df_final[new_col] = df_filtered[raw_col]
        else:
            # ใช้ regex ล้างเครื่องหมายจุลภาคออก และแปลงเป็นค่า Int
            df_final[new_col] = df_filtered[raw_col].astype(str).str.replace(',', '', regex=False).str.strip()
            df_final[new_col] = pd.to_numeric(df_final[new_col], errors='coerce').fillna(0).astype(int)

print("\n--- 📊 🎉 สรุปดัชนีสิ่งปลูกสร้างจดทะเบียนจริงทั่วกรุงเทพฯ ---")
print(f"• จำนวนเขตที่เคลียร์สะอาด: {df_final['district'].nunique()} เขตยุทธศาสตร์ครบถ้วน")
print(f"• ยอดรวมยูนิตอาคารชุด (คอนโด): {df_final['condo'].sum():,} ยูนิต (ทำเลทองคนรุ่นใหม่)")
print(f"• ยอดรวมบ้านแนวราบ (บ้านเดี่ยว+ทาวน์เฮ้าส์): {(df_final['house'].sum() + df_final['townhouse'].sum()):,} หลัง")
print(f"• ยอดรวมพื้นที่สำนักงาน/ออฟฟิศทำงาน: {df_final['office'].sum():,} แห่ง")

# 4. บันทึกเดตาสะอาดก้อนใหม่เข้าคลังสินทรัพย์ปลายทาง
os.makedirs("../../data/interim", exist_ok=True)
df_final.to_json(output_file_path, orient='records', force_ascii=False, indent=4)
print(f"\n💾 🎉 บันทึก Asset ตัวใหม่สำเร็จ! จัดเก็บไฟล์ไว้ที่: {output_file_path}")

# 5. 🧹 สั่ง Garbage Collection เคลียร์ขยะออกจาก RAM ทันที
if 'df_raw' in locals(): del df_raw
if 'df_filtered' in locals(): del df_filtered
if 'df_final' in locals(): del df_final
gc.collect()
print("🤖 [สถานะ: ปลอดภัย 100%] ทำความสะอาดแรมเครื่องเรียบร้อย เบาหวิวพร้อมทำงานต่อครับ!")

1. ⚙️ กำลังโหลดข้อมูลสิ่งปลูกสร้างจดทะเบียน (ปรับ skiprows=4 เพื่อแก้บั๊ก KeyError)...
2. 🧹 กำลังทำความสะอาดล้างแถวสรุปยอดและล้างชื่อเขตให้เป็นมาตรฐาน...
3. 🔢 แปลงฟีเจอร์ตัวเลข String (ที่มีคอมมา) ให้กลายเป็นประเภท Integer...

--- 📊 🎉 สรุปดัชนีสิ่งปลูกสร้างจดทะเบียนจริงทั่วกรุงเทพฯ ---
• จำนวนเขตที่เคลียร์สะอาด: 50 เขตยุทธศาสตร์ครบถ้วน
• ยอดรวมยูนิตอาคารชุด (คอนโด): 889,823 ยูนิต (ทำเลทองคนรุ่นใหม่)
• ยอดรวมบ้านแนวราบ (บ้านเดี่ยว+ทาวน์เฮ้าส์): 2,070,244 หลัง
• ยอดรวมพื้นที่สำนักงาน/ออฟฟิศทำงาน: 10,979 แห่ง

💾 🎉 บันทึก Asset ตัวใหม่สำเร็จ! จัดเก็บไฟล์ไว้ที่: ../cleaned-assets/bangkok_housing_clean.json
🤖 [สถานะ: ปลอดภัย 100%] ทำความสะอาดแรมเครื่องเรียบร้อย เบาหวิวพร้อมทำงานต่อครับ!


,ข้อมูลที่อยู่อาศัยสร้างเสร็จจดทะเบียน เดือน พฤษภาคม 2569,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,การแสดงผลจังหวัด แยกตามอำเภอ (กรุงเทพมหานคร),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ลำดับที่,จังหวัด/อำเภอ,บ้าน (1),โรงแรม (7),โรงงาน (13),หอพัก (14),ตึกแถว (16),ห้องแถว (17),ตึก (18),ร้านค้า (19),...,ห้างหุ้นส่วนจำกัด (33),ห้างหุ้นส่วนนิติบุคคล (34),โกดัง (37),บริษัท (38),ทาวน์เฮ้าส์ (39),บ้านแฝด (45),บ้านแถว (49),ทาวน์โฮม (51),อื่น ๆ,รวม
4,NaN,รวม,"1,969,244","1,778",656,"2,193","52,671","1,057","106,609","12,488",...,23,5,792,558,"101,000","22,662","38,586",765,"107,746","3,373,689"
5,NaN,กรุงเทพมหานคร,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1,ท้องถิ่นเขตพระนคร,"16,602",116,0,2,11,0,16,775,...,0,0,0,1,0,0,0,0,697,"18,771"
7,2,ท้องถิ่นเขตดุสิต,"19,570",4,9,6,110,21,431,4,...,0,0,0,1,23,0,0,0,"9,175","34,408"
8,3,ท้องถิ่นเขตหนองจอก,"53,799",1,23,12,774,0,"12,090",250,...,0,0,52,8,"3,154","2,079",506,0,"1,993","76,434"
9,4,ท้องถิ่นเขตบางรัก,"20,250",112,0,4,369,17,"5,236",48,...,0,0,0,0,0,0,0,0,361,"36,130"
